# BERT 모델 Fine-tuning 실습 정리

이 노트북은 **Hugging Face Model Hub**, **Transformers 설치**, **Pre-trained BERT 모델 다운로드**, **BERT 내부 구조 확인**, **Fine-tuning 범위 설정** 내용을 코드로 실습하도록 구성한 파일입니다.

BERT는 이미 대규모 말뭉치로 사전 학습된 언어 모델입니다. 실무에서는 처음부터 BERT를 새로 학습하지 않고, Hugging Face에서 제공하는 사전 학습 모델을 다운로드한 뒤 필요한 작업에 맞게 일부 또는 전체 파라미터를 다시 학습하는 Fine-tuning 방식을 주로 사용합니다.

## 1. Transformers 라이브러리 설치

강의교안에서는 Hugging Face Transformers를 Python 패키지로 설치한 뒤 BERT 계열 모델을 다운로드해 사용하는 흐름을 설명합니다. 아래 코드는 Google Colab이나 Jupyter 환경에서 필요한 패키지를 설치합니다.

In [1]:
# Hugging Face Transformers 라이브러리를 설치합니다.
# - transformers: BERT, GPT 등 사전 학습 언어 모델을 쉽게 불러오고 사용할 수 있게 해 주는 라이브러리입니다.
# - accelerate: Trainer 또는 PyTorch 학습 과정에서 장치 관리와 학습 실행을 안정적으로 도와주는 라이브러리입니다.
# - -q 옵션은 설치 로그를 간단하게 출력하기 위한 옵션입니다.
!pip install -q transformers accelerate


## 2. BERT Sequence Classification 모델 다운로드

 `BertForSequenceClassification` 처럼, 문장 분류 작업에는 BERT 본체 위에 분류용 출력층이 붙은 모델 클래스를 사용합니다. `from_pretrained()` 함수는 지정한 모델명을 기준으로 Hugging Face Hub에서 설정 파일과 가중치를 다운로드합니다.

In [2]:
# BERT 기반 문장 분류 모델 클래스를 불러옵니다.
# BertForSequenceClassification은 BERT 인코더 뒤에 분류용 Linear Layer가 추가된 모델입니다.
from transformers import BertForSequenceClassification

# 다운로드할 사전 학습 BERT 모델명을 지정합니다.
# bert-base-uncased는 영어 소문자 기반 BERT Base 모델입니다.
# - base: BERT 기본 크기 모델입니다.
# - uncased: 대문자와 소문자를 구분하지 않고 모두 소문자처럼 처리합니다.
bert_model_name = "bert-base-uncased"

# Hugging Face Hub에서 사전 학습된 BERT 분류 모델을 다운로드합니다.
# num_labels=2는 이 모델을 이진 분류용으로 사용하겠다는 의미입니다.
# 예: 긍정/부정, 정상/비정상, 스팸/정상 등 두 개 클래스를 예측합니다.
model = BertForSequenceClassification.from_pretrained(
    bert_model_name,
    num_labels=2
)

# 다운로드된 모델 객체를 출력하여 전체 구조를 간략히 확인합니다.
model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## 3. BERT 내부 Layer 이름 확인

`model.bert.named_parameters()`를 이용해 BERT 내부 파라미터 이름을 확인합니다. 이름을 확인해야 Fine-tuning할 층과 동결할 층을 구분할 수 있습니다. BERT Base는 보통 Embedding Layer와 Encoder Layer 12개로 구성됩니다.

In [3]:
# BERT 본체 내부의 파라미터 이름만 확인합니다.
# named_parameters()는 (파라미터 이름, 파라미터 텐서) 쌍을 차례대로 반환합니다.
# 여기서는 전체 텐서를 출력하지 않고 이름만 출력하여 구조를 빠르게 확인합니다.
for name, param in model.bert.named_parameters():
    # name에는 embeddings, encoder.layer.0, encoder.layer.1 같은 계층 이름이 들어 있습니다.
    print(name)


embeddings.word_embeddings.weight
embeddings.position_embeddings.weight
embeddings.token_type_embeddings.weight
embeddings.LayerNorm.weight
embeddings.LayerNorm.bias
encoder.layer.0.attention.self.query.weight
encoder.layer.0.attention.self.query.bias
encoder.layer.0.attention.self.key.weight
encoder.layer.0.attention.self.key.bias
encoder.layer.0.attention.self.value.weight
encoder.layer.0.attention.self.value.bias
encoder.layer.0.attention.output.dense.weight
encoder.layer.0.attention.output.dense.bias
encoder.layer.0.attention.output.LayerNorm.weight
encoder.layer.0.attention.output.LayerNorm.bias
encoder.layer.0.intermediate.dense.weight
encoder.layer.0.intermediate.dense.bias
encoder.layer.0.output.dense.weight
encoder.layer.0.output.dense.bias
encoder.layer.0.output.LayerNorm.weight
encoder.layer.0.output.LayerNorm.bias
encoder.layer.1.attention.self.query.weight
encoder.layer.1.attention.self.query.bias
encoder.layer.1.attention.self.key.weight
encoder.layer.1.attention.self.key

## 4. 파라미터 개수와 학습 가능 여부 확인

Fine-tuning에서는 모든 파라미터를 학습할 수도 있고, 일부 Layer만 학습할 수도 있습니다. `requires_grad=True`이면 학습 중 파라미터가 업데이트되고, `requires_grad=False`이면 해당 파라미터가 고정됩니다.

In [4]:
# 전체 파라미터 수와 학습 가능한 파라미터 수를 계산하는 함수를 정의합니다.
def count_parameters(model):
    # 모든 파라미터 원소 개수를 합산합니다.
    total_params = sum(param.numel() for param in model.parameters())

    # requires_grad=True인 파라미터만 학습 가능한 파라미터로 계산합니다.
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)

    # 계산 결과를 딕셔너리 형태로 반환합니다.
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "frozen_params": total_params - trainable_params
    }

# 현재 모델의 파라미터 상태를 확인합니다.
count_parameters(model)


{'total_params': 109483778, 'trainable_params': 109483778, 'frozen_params': 0}

## 5. BERT Layer별 파라미터 이름과 Shape 확인

다운로드 모델 구조 확인 단계입니다. 실제 파라미터 값을 전부 출력하면 양이 매우 많으므로, 실습에서는 이름과 Shape를 확인하는 방식이 더 안전합니다.

In [5]:
# BERT 본체의 파라미터 이름, 텐서 크기, 학습 가능 여부를 보기 좋게 출력합니다.
for name, param in model.bert.named_parameters():
    # tuple(param.shape)는 해당 파라미터 텐서의 차원을 의미합니다.
    # param.requires_grad는 해당 파라미터가 학습 대상인지 여부를 의미합니다.
    print(f"{name:70s} shape={tuple(param.shape)} requires_grad={param.requires_grad}")


embeddings.word_embeddings.weight                                      shape=(30522, 768) requires_grad=True
embeddings.position_embeddings.weight                                  shape=(512, 768) requires_grad=True
embeddings.token_type_embeddings.weight                                shape=(2, 768) requires_grad=True
embeddings.LayerNorm.weight                                            shape=(768,) requires_grad=True
embeddings.LayerNorm.bias                                              shape=(768,) requires_grad=True
encoder.layer.0.attention.self.query.weight                            shape=(768, 768) requires_grad=True
encoder.layer.0.attention.self.query.bias                              shape=(768,) requires_grad=True
encoder.layer.0.attention.self.key.weight                              shape=(768, 768) requires_grad=True
encoder.layer.0.attention.self.key.bias                                shape=(768,) requires_grad=True
encoder.layer.0.attention.self.value.weight          

## 6. Fine-tuning 전략 설정

`tl_strategy` 값에 따라 학습 범위를 다르게 설정합니다.

- `1`: BERT 본체 전체를 동결하고 분류층만 학습
- `2`: BERT의 Pooler와 분류층 중심으로 학습
- `3`: BERT의 마지막 Encoder Layer와 Pooler, 분류층만 학습

BERT Base는 Encoder Layer가 `0~11`까지 있으므로 마지막 Layer는 `layer.11`입니다.

In [6]:
# Fine-tuning 전략을 선택합니다.
# 1: BERT 본체 전체 동결
# 2: Pooler를 제외한 BERT 본체 대부분 동결
# 3: 마지막 Encoder Layer와 Pooler만 학습 가능하도록 설정
tl_strategy = 3

# 먼저 모든 파라미터를 학습 가능 상태로 초기화합니다.
# 여러 번 실행해도 이전 설정이 남지 않도록 하기 위한 안전 처리입니다.
for param in model.parameters():
    param.requires_grad = True

# BERT Base의 마지막 Encoder Layer 이름입니다.
# bert-base-uncased는 encoder.layer.0부터 encoder.layer.11까지 총 12개 Encoder Layer를 가집니다.
last_encoder_layer_name = "encoder.layer.11"

if tl_strategy == 1:
    # 전략 1: BERT 본체 전체를 동결합니다.
    # 분류층 classifier는 model.bert 내부가 아니므로 계속 학습 가능합니다.
    for name, param in model.bert.named_parameters():
        param.requires_grad = False

elif tl_strategy == 2:
    # 전략 2: pooler를 제외한 BERT 본체 파라미터를 동결합니다.
    # pooler는 [CLS] 토큰 표현을 문장 단위 표현으로 바꾸는 데 사용됩니다.
    for name, param in model.bert.named_parameters():
        if not name.startswith("pooler"):
            param.requires_grad = False

elif tl_strategy == 3:
    # 전략 3: 마지막 Encoder Layer와 pooler를 제외한 BERT 본체 파라미터를 동결합니다.
    # 마지막 Layer만 학습하면 전체 학습보다 빠르고, 데이터가 적을 때 과적합을 줄이는 데 도움이 됩니다.
    for name, param in model.bert.named_parameters():
        if (not name.startswith("pooler")) and (last_encoder_layer_name not in name):
            param.requires_grad = False

else:
    # 정의하지 않은 전략 값이 들어오면 명확한 오류를 발생시킵니다.
    raise ValueError("tl_strategy는 1, 2, 3 중 하나로 설정해야 합니다.")

# 설정 후 전체 파라미터 수와 학습 가능한 파라미터 수를 다시 확인합니다.
count_parameters(model)


{'total_params': 109483778,
 'trainable_params': 7680002,
 'frozen_params': 101803776}

## 7. Fine-tuning 적용 결과 확인

아래 코드는 각 파라미터가 학습 가능한 상태인지 확인합니다. `requires_grad=False`로 표시되는 파라미터는 학습 중 업데이트되지 않습니다.

In [7]:
# Fine-tuning 전략 적용 후 BERT 본체의 파라미터 학습 여부를 출력합니다.
for name, param in model.bert.named_parameters():
    # True이면 학습 대상, False이면 동결 대상입니다.
    print(f"{name:70s} requires_grad={param.requires_grad}")

# 분류층은 BERT 본체 밖에 있으므로 별도로 확인합니다.
for name, param in model.classifier.named_parameters():
    print(f"classifier.{name:59s} requires_grad={param.requires_grad}")


embeddings.word_embeddings.weight                                      requires_grad=False
embeddings.position_embeddings.weight                                  requires_grad=False
embeddings.token_type_embeddings.weight                                requires_grad=False
embeddings.LayerNorm.weight                                            requires_grad=False
embeddings.LayerNorm.bias                                              requires_grad=False
encoder.layer.0.attention.self.query.weight                            requires_grad=False
encoder.layer.0.attention.self.query.bias                              requires_grad=False
encoder.layer.0.attention.self.key.weight                              requires_grad=False
encoder.layer.0.attention.self.key.bias                                requires_grad=False
encoder.layer.0.attention.self.value.weight                            requires_grad=False
encoder.layer.0.attention.self.value.bias                              requires_grad=False